# Previsão de temperatura — Regressão (mês + localização)

Adaptação do **Ex05_03**, mas em **regressão**: prever `temp_media` a partir do **mês** (codificado em seno/cosseno, para
apanhar a sazonalidade) e da **localização** (`lat`, `lon`). Avalia com **RMSE** e **R²**, e
compara `LinearRegression` com `RandomForestRegressor`.
Nota: com ~17 meses de dados, isto é uma demonstração — não um modelo de produção.

In [17]:
import math
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import ipywidgets as widgets
from ipywidgets import interact

import pandas as pd
import matplotlib.pyplot as plt

import calendar
import numpy as np


spark = SparkSession.builder.appName("PrevTemp").getOrCreate()

GOLD_DAILY = "hdfs://hadoop-master:39000/2250055/gold/gold_daily"

# Carrega o gold diario e cria as features:
#  - mes_sin / mes_cos : mes (1-12) em forma ciclica, para a regressao apanhar a sazonalidade
#  - lat / lon         : localizacao do distrito
df = (spark.read.parquet(GOLD_DAILY)
      .withColumn("mes", F.month("date"))
      .withColumn("mes_sin", F.sin(2 * math.pi * F.col("mes") / 12))
      .withColumn("mes_cos", F.cos(2 * math.pi * F.col("mes") / 12))
      .select("district", "date", "lat", "lon", "mes_sin", "mes_cos", "temp_media")
      .na.drop())

df.show(5, truncate=False)

+----------------+----------+----+------+-------------------+------------------+----------+
|district        |date      |lat |lon   |mes_sin            |mes_cos           |temp_media|
+----------------+----------+----+------+-------------------+------------------+----------+
|Viana do Castelo|2025-01-01|41.7|-8.833|0.49999999999999994|0.8660254037844387|10.0      |
|Viana do Castelo|2025-01-02|41.7|-8.833|0.49999999999999994|0.8660254037844387|11.0      |
|Viana do Castelo|2025-01-03|41.7|-8.833|0.49999999999999994|0.8660254037844387|12.0      |
|Viana do Castelo|2025-01-04|41.7|-8.833|0.49999999999999994|0.8660254037844387|14.0      |
|Viana do Castelo|2025-01-05|41.7|-8.833|0.49999999999999994|0.8660254037844387|14.0      |
+----------------+----------+----+------+-------------------+------------------+----------+
only showing top 5 rows



In [13]:
# Monta o vetor de features e separa treino/teste (70/30)
assembler = VectorAssembler(
    inputCols=["mes_sin", "mes_cos", "lat", "lon"],
    outputCol="features")

train, test = df.randomSplit([0.7, 0.3], seed=42)
print("treino:", train.count(), "| teste:", test.count())

treino: 6687 | teste: 2745


In [14]:
# Treina e avalia dois modelos (como o Ex05_03 comparava algoritmos)
rmse = RegressionEvaluator(labelCol="temp_media", predictionCol="prediction", metricName="rmse")
r2   = RegressionEvaluator(labelCol="temp_media", predictionCol="prediction", metricName="r2")

modelos = {
    "LinearRegression": LinearRegression(labelCol="temp_media", featuresCol="features"),
    "RandomForest":     RandomForestRegressor(labelCol="temp_media", featuresCol="features", numTrees=50),
}

resultados = {}
for nome, alg in modelos.items():
    modelo = Pipeline(stages=[assembler, alg]).fit(train)
    pred = modelo.transform(test)
    resultados[nome] = (modelo, pred)
    print(f"{nome:18s}  RMSE = {rmse.evaluate(pred):5.2f} C   R2 = {r2.evaluate(pred):.3f}")

26/06/09 22:28:31 WARN Instrumentation: [5e7a0e6a] regParam is zero, which might cause numerical instability and overfitting.


LinearRegression    RMSE =  3.33 C   R2 = 0.699
RandomForest        RMSE =  3.01 C   R2 = 0.754


In [15]:
# Espreita previsoes vs valores reais (modelo RandomForest)
_, pred_lr = resultados["RandomForest"]
pred_lr.select("district", "date", "temp_media",
               F.round("prediction", 1).alias("previsto")).show(10, truncate=False)

+--------+----------+----------+--------+
|district|date      |temp_media|previsto|
+--------+----------+----------+--------+
|Aveiro  |2025-01-03|12.0      |10.8    |
|Aveiro  |2025-01-07|13.0      |10.8    |
|Aveiro  |2025-01-09|14.0      |10.8    |
|Aveiro  |2025-01-10|14.0      |10.8    |
|Aveiro  |2025-01-14|10.0      |10.8    |
|Aveiro  |2025-01-15|8.0       |10.8    |
|Aveiro  |2025-01-16|9.0       |10.8    |
|Aveiro  |2025-01-20|11.0      |10.8    |
|Aveiro  |2025-01-22|14.0      |10.8    |
|Aveiro  |2025-01-24|13.0      |10.8    |
+--------+----------+----------+--------+
only showing top 10 rows



In [20]:

# modelo usado para a linha de previsao (podes trocar para "LinearRegression")
modelo_plot, _ = resultados["RandomForest"]

# 1) previsao por (distrito, mes) - calculada uma vez
meses_df = spark.createDataFrame([(m,) for m in range(1, 13)], ["mes"])
grid = (df.select("district", "lat", "lon").distinct().crossJoin(meses_df)
          .withColumn("mes_sin", F.sin(2*math.pi*F.col("mes")/12))
          .withColumn("mes_cos", F.cos(2*math.pi*F.col("mes")/12)))
preds = {(r["district"], r["mes"]): r["prediction"]
         for r in modelo_plot.transform(grid).select("district", "mes", "prediction").collect()}

# 2) temperaturas diarias reais - trazidas uma vez para pandas
diario = (df.select("district", "date", "temp_media")
            .withColumn("mes", F.month("date"))
            .toPandas())
diario["date"] = pd.to_datetime(diario["date"])
diario["dia"]  = diario["date"].dt.day
diario["ano"]  = diario["date"].dt.year

distritos = sorted(diario["district"].unique().tolist())
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]
NOME_MES = {v: k for k, v in MESES}

def mostrar_plot(distrito, mes):
    d = diario[(diario["district"] == distrito) & (diario["mes"] == mes)]
    previsto = preds[(distrito, mes)]

    plt.figure(figsize=(9, 5))
    for ano, g in d.sort_values("dia").groupby("ano"):
        plt.plot(g["dia"], g["temp_media"], marker="o", label=f"real {ano}")
    plt.axhline(previsto, color="r", linestyle="--", label=f"previsto ({previsto:.1f} °C)")
    plt.xlabel("Dia do mês"); plt.ylabel("Temperatura média (°C)")
    plt.title(f"{distrito} — {NOME_MES[mes]}: real (diário) vs previsto")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.show()

interact(mostrar_plot,
         distrito=widgets.Dropdown(options=distritos, description="Distrito:"),
         mes=widgets.Dropdown(options=MESES, value=8, description="Mês:"));

interactive(children=(Dropdown(description='Distrito:', options=('Aveiro', 'Beja', 'Braga', 'Braganca', 'Caste…

In [19]:
modelo_lr, _ = resultados["RandomForest"]

# opções: distritos a partir dos dados, e meses com nome
distritos = [r["district"] for r in df.select("district").distinct().orderBy("district").collect()]
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]

def prever_temp(distrito, mes):
    loc = df.filter(F.col("district") == distrito).select("lat", "lon").first()
    novo = spark.createDataFrame(
        [(math.sin(2*math.pi*mes/12), math.cos(2*math.pi*mes/12), loc["lat"], loc["lon"])],
        ["mes_sin", "mes_cos", "lat", "lon"])
    prev = modelo_lr.transform(novo).select(F.round("prediction", 1).alias("p")).first()[0]
    print(f"Temperatura média prevista para {distrito}, mês {mes}: {prev} °C")

interact(prever_temp,
         distrito=widgets.Dropdown(options=distritos, description="Distrito:"),
         mes=widgets.Dropdown(options=MESES, value=8, description="Mês:"));

interactive(children=(Dropdown(description='Distrito:', options=('Aveiro', 'Beja', 'Braga', 'Braganca', 'Caste…

In [ ]:
spark.stop()